# Advanced 01 — Algorithm Design and Graphs

Apply graph thinking to ordering problems and shortest paths. Even though this is 'advanced', the hints walk you through the algorithm shapes.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Setup Cell — Run this first! (Don't worry if you don't understand everything yet)
# ═══════════════════════════════════════════════════════════════

# What is 'import'? It loads code from other files so we can use helpful functions.
# Think of it like borrowing tools from a toolbox instead of building everything from scratch.

import os   # 'os' = operating system tools (helps us work with files and folders)
import sys  # 'sys' = system tools (lets us modify how Python finds our code)

# This next part helps Python find our helper functions in the 'utils' folder.
# You don't need to understand HOW it works yet — just run this cell before starting!

PROJECT_ROOT = os.path.abspath(os.path.join("..", ".."))  # Find the main project folder
if PROJECT_ROOT not in sys.path:      # Check if Python knows about this folder
    sys.path.append(PROJECT_ROOT)     # If not, tell Python to look there for code

# Now import our testing helper functions from the utils folder
from utils.validation import check_equal, check_true, summary, reset

# Reset the test counter (so we start fresh each time we run the notebook)
reset()

**How to use this notebook**

- Read the problem and the step-by-step hints first. They are written for a motivated first-time coder.
- Look for **Show/hide** sections; click to reveal extra guidance or answers when you feel stuck.
- Write your solution code directly below each TODO, keeping functions short and clear.
- Run the tests cell; if something fails, re-read the hint and add small `print` checks to see what is happening.
- When all checks pass, add one or two of your own test cases to prove you really understand it.


---

## 📚 Quick Lesson: Graph Theory Basics

**Graphs** are one of the most powerful concepts in computer science. They model relationships and connections between things.

### What is a Graph?

A **graph** consists of:
- **Nodes** (also called "vertices") - the things
- **Edges** (also called "links" or "connections") - the relationships between things

**Real-world examples:**
- **Social network**: Nodes = people, Edges = friendships
- **Map**: Nodes = cities, Edges = roads
- **Course prerequisites**: Nodes = classes, Edges = "must take before"
- **Web pages**: Nodes = pages, Edges = links between pages

### Visual Representation

```
Simple graph with 4 nodes:

    A -------- B
    |          |
    |          |
    C -------- D

Nodes: A, B, C, D
Edges: A-B, A-C, B-D, C-D
```

### Types of Graphs

**1. Directed Graph (Digraph)**
Edges have a direction (like one-way streets)

```
    A -----→ B
    ↓        ↓
    C ←----- D

A→B (A points to B)
A→C
D→C
B→D
```

**Example:** Course prerequisites
- "Must take A before B" → edge from A to B
- **Indegree** = number of edges coming IN (prerequisites)
- **Outdegree** = number of edges going OUT (classes that need this one)

**2. Undirected Graph**
Edges have no direction (like two-way streets)

```
    A ----- B
    |       |
    C ----- D

A and B are connected (can go either way)
```

**Example:** Facebook friendships (if A is friends with B, then B is friends with A)

### Representing Graphs in Code

**Method 1: Edge List**
Just list all the edges as pairs:

```python
# Directed graph
edges = [(0, 1), (0, 2), (1, 3), (2, 3)]
# Means: 0→1, 0→2, 1→3, 2→3
```

**Method 2: Adjacency List** (most common!)
For each node, list its neighbors:

```python
# Directed graph - for each node, list where it points to
graph = {
    0: [1, 2],  # Node 0 has edges to nodes 1 and 2
    1: [3],     # Node 1 has edge to node 3
    2: [3],     # Node 2 has edge to node 3
    3: []       # Node 3 has no outgoing edges
}

# Check what neighbors node 0 has
neighbors = graph[0]  # [1, 2]
```

**Method 3: Adjacency Matrix**
2D array where `matrix[i][j] = 1` if edge from i to j:

```python
# For 4 nodes (0, 1, 2, 3)
matrix = [
    [0, 1, 1, 0],  # Node 0 → nodes 1, 2
    [0, 0, 0, 1],  # Node 1 → node 3
    [0, 0, 0, 1],  # Node 2 → node 3
    [0, 0, 0, 0]   # Node 3 → nothing
]

# Check if edge from 0 to 2
has_edge = matrix[0][2] == 1  # True
```

### Important Graph Concepts for Tasks

**Indegree (Directed Graphs)**
Number of edges **coming into** a node.

```
    A -----→ B ←----- C
             ↓
             D

Indegrees:
- A: 0 (nothing points to A)
- B: 2 (A and C point to B)
- D: 1 (only B points to D)
```

**Why it matters for Task 1:**
- Nodes with indegree 0 have no prerequisites - you can do them first!
- As you "complete" a task, decrement indegrees of things that depend on it

**Path**
A sequence of nodes where each consecutive pair is connected by an edge.

```
    A → B → C → D

Path from A to D: [A, B, C, D]
Path length: 3 (number of edges)
```

**Shortest Path**
The path with the fewest edges between two nodes.

```
    A → B → D
    ↓       ↑
    C ------+

Paths from A to D:
- A → B → D (length 2) ← shortest!
- A → C → D (length 2) ← also shortest!
```

**Why BFS finds shortest paths:**
- BFS explores in "layers" by distance
- Layer 0: starting node
- Layer 1: nodes 1 edge away
- Layer 2: nodes 2 edges away
- The first time you reach the goal = minimum distance!

**Cycle**
A path that starts and ends at the same node.

```
    A → B
    ↑   ↓
    D ← C

Cycle: A → B → C → D → A
```

**Why cycles matter for Task 1:**
- If courses have circular dependencies (A needs B, B needs A), you can NEVER complete them!
- Topological sort only works if there are NO cycles

### Graph Traversal Algorithms

**Breadth-First Search (BFS)**
Explore layer by layer (closest first).

```python
from collections import deque

def bfs(graph, start):
    """Visit all reachable nodes, layer by layer."""
    queue = deque([start])
    visited = {start}
    
    while queue:
        node = queue.popleft()  # Get oldest (FIFO)
        print(f"Visiting: {node}")
        
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
```

**Visual BFS:**
```
Graph:      0
           / \
          1   2
         / \
        3   4

BFS order: 0, 1, 2, 3, 4
(Level 0: 0)
(Level 1: 1, 2)
(Level 2: 3, 4)
```

**Use BFS when:**
- You want the shortest path (Task 2!)
- You want to explore "nearby" things first

**Depth-First Search (DFS)**
Explore as deep as possible before backtracking.

```python
def dfs(graph, start):
    """Visit all reachable nodes, going deep first."""
    stack = [start]
    visited = {start}
    
    while stack:
        node = stack.pop()  # Get newest (LIFO)
        print(f"Visiting: {node}")
        
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                stack.append(neighbor)
```

**Visual DFS:**
```
Graph:      0
           / \
          1   2
         / \
        3   4

DFS order: 0, 1, 3, 4, 2
(Plunge deep: 0 → 1 → 3, backtrack, → 4, backtrack, → 2)
```

### Topological Sort (Task 1)

**What it does:** Orders nodes so all edges point "forward" (no node comes before its prerequisites).

**Example:**
```
Courses:    A → C
            ↓   ↓
            B → D

Prerequisites:
- Must take A before C
- Must take A before B  
- Must take B before D
- Must take C before D

Valid orderings:
- [A, B, C, D] ✓
- [A, C, B, D] ✓

Invalid:
- [B, A, C, D] ✗ (B before A, but A is prerequisite!)
```

**Kahn's Algorithm (what you'll implement):**
1. Find all nodes with indegree 0 (no prerequisites)
2. Add one to your ordering
3. "Remove" that node (decrement indegree of its neighbors)
4. Repeat until done

**Visual:**
```
Start:  A → B    Indegrees: A=0, B=1, C=2
        ↓   ↓
        C ←-+

Step 1: A has indegree 0 → pick A
Order: [A]
Update indegrees: B=0, C=1 (decreased because A is "done")

Step 2: B has indegree 0 → pick B
Order: [A, B]
Update indegrees: C=0

Step 3: C has indegree 0 → pick C
Order: [A, B, C]

Done! ✓
```

### Greedy Algorithms (Task 3)

**Greedy** means making the locally optimal choice at each step, hoping it leads to a global optimum.

**Example: Earliest Deadline First**
- Always pick the task with the earliest deadline next
- Hope that minimizing lateness locally minimizes it overall

```python
tasks = [
    {"name": "Essay", "deadline": 5, "duration": 3},
    {"name": "Quiz", "deadline": 2, "duration": 1},
    {"name": "Lab", "deadline": 3, "duration": 2}
]

# Greedy choice: earliest deadline first
# Sort by deadline: Quiz (2), Lab (3), Essay (5)
# Schedule: [Quiz, Lab, Essay]
```

**Does greedy always work?**
- For some problems (like this one): **Yes!** ✓
- For others (like the 0/1 knapsack): **No!** ✗
- The trick is knowing when greedy is safe

---

Now you're ready to tackle graph algorithms! You'll implement topological sort (scheduling), BFS shortest path (navigation), and greedy scheduling (optimization).

### Task 1 — Topological sort

Write `topological_sort(num_nodes, edges)` where nodes are labeled `0..num_nodes-1` and `edges` is a list of `(u, v)` pairs meaning `u` must come before `v`.
Return a valid ordering list or raise `ValueError` if the graph has a cycle.

**Why this matters:** Scheduling with dependencies shows up everywhere.

**Step-by-step hint (Kahn's algorithm):**
1) Compute indegrees (how many prerequisites) for each node.
2) Start a queue with all nodes that have indegree 0.
3) Repeatedly pop from the queue, add to ordering, and reduce indegree of its outgoing neighbors.
4) If a neighbor's indegree hits 0, push it into the queue.
5) At the end, if ordering length is num_nodes, return it; else raise `ValueError` for a cycle.


### Task 2 — Shortest path in a grid

Write `shortest_path_grid(grid, start, goal)` that returns the length (number of steps) of the shortest path in a 0/1 grid.
Return `None` if no path exists. Use BFS.

**Why this matters:** Breadth-first search (BFS) finds shortest paths in unweighted graphs.

**Step-by-step hint:**
1) Use a queue storing `(position, distance_so_far)`.
2) Begin with `start` at distance 0; mark it visited.
3) Pop from the queue, and if it's the goal, return the distance.
4) Otherwise, push all open, in-bounds neighbors that are not visited, with distance + 1.
5) If the queue empties without reaching goal, return `None`.


### Task 3 — Greedy scheduling by deadlines

Write `schedule_by_deadline(tasks)` where each task is a dict with keys:
- `name` (string)
- `duration` (int)
- `deadline` (int)

Use an **earliest deadline first** strategy; on ties, pick the shorter duration first.
Return an ordered list of task names.

**Why this matters:** Greedy rules are simple to implement if stated clearly.

**Step-by-step hint:**
1) Sort the tasks by deadline ascending, then by duration ascending (two-key sort).
2) After sorting, return a list of the task names in that order.


In [ ]:
def topological_sort(num_nodes, edges):
    """Apply Kahn's algorithm: peel off indegree-0 nodes to produce a valid ordering or detect a cycle."""
    # Step 1: build indegree counts for each node.
    # Step 2: queue all nodes with indegree 0.
    # Step 3: pop from queue, add to ordering, decrement neighbors' indegrees.
    # Step 4: push new indegree-0 neighbors to queue.
    # Step 5: if ordering length == num_nodes, return it; else raise ValueError for a cycle.
    raise NotImplementedError


def shortest_path_grid(grid, start, goal):
    """Use BFS layers to find the minimum steps from start to goal in an unweighted grid."""
    # Step 1: queue holds (position, distance); start at distance 0.
    # Step 2: while queue not empty, pop; if goal, return distance.
    # Step 3: add valid, not-yet-visited open neighbors with distance+1.
    # Step 4: return None if queue empties without reaching goal.
    raise NotImplementedError


def schedule_by_deadline(tasks):
    """Greedily sort by deadline (then duration) to produce a reasonable execution order quickly."""
    # Step 1: sort tasks by (deadline, duration).
    # Step 2: return a list of task names in that sorted order.
    raise NotImplementedError


### Checkpoint — verify ordering and paths

Run the next cell. If something fails, print your indegree table for topo sort or the queue contents for BFS to see where logic diverges.


In [ ]:
reset()

order = topological_sort(4, [(0, 1), (0, 2), (1, 3), (2, 3)])
check_true(
    "topological_sort respects edges",
    order.index(0) < order.index(1) and order.index(0) < order.index(2) and order.index(1) < order.index(3) and order.index(2) < order.index(3),
)

grid = [
    [0, 0, 0],
    [1, 1, 0],
    [0, 0, 0],
]
check_equal("shortest_path_grid reachable", shortest_path_grid(grid, (0, 0), (2, 2)), 5)
check_equal("shortest_path_grid blocked", shortest_path_grid([[0, 1], [1, 0]], (0, 0), (1, 1)), None)

tasks = [
    {"name": "A", "duration": 3, "deadline": 5},
    {"name": "B", "duration": 2, "deadline": 3},
    {"name": "C", "duration": 1, "deadline": 3},
]
check_equal("schedule_by_deadline", schedule_by_deadline(tasks), ["B", "C", "A"])
summary()


### Interview warm-up (click to reveal answers)

<details>
<summary>How do you detect a cycle in topological sorting?</summary>
If you process nodes with indegree 0 and the final ordering has fewer nodes than the graph, a cycle prevented some nodes from ever reaching indegree 0.
</details>

<details>
<summary>Why does BFS give shortest path length in an unweighted grid?</summary>
BFS explores in layers by distance; the first time you reach the goal is guaranteed to be the minimum number of steps.
</details>

<details>
<summary>What is a greedy choice in scheduling by deadline?</summary>
Always pick the available task with the earliest deadline (and shorter duration on ties) to minimize lateness locally, which leads to a good global order here.
</details>
